In [ ]:
#imports
import numpy as np
import pandas as pd
import io
import time
import heapq
import random
import matplotlib.pyplot as plt
from statistics import mean
from collections import deque

In [ ]:
# Data source: DOT Consumer Airfare Report - Table 1a (All U.S. Airport Pair Markets).
# Download table1a.csv from https://datahub.transportation.gov/d/tfrh-tu9e
# and place it in the same folder as this notebook.

In [ ]:
#load Table 1a (DOT Consumer Airfare Report) from a local file
flight_data = pd.read_csv('table1a.csv')

# Table 1a stores numbers as formatted strings (e.g. $132.68 or 1,138); convert to numeric
for c in ['fare', 'fare_lg', 'fare_low']:
    flight_data[c] = flight_data[c].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
for c in ['nsmiles', 'passengers']:
    flight_data[c] = flight_data[c].astype(str).str.replace(',', '', regex=False).astype(int)

In [ ]:
flight_data

In [ ]:
#keep only the columns needed for the graph (works for both the Kaggle and Table 1a exports)
keep = ['Year', 'quarter', 'city1', 'city2', 'airport_1', 'airport_2', 'nsmiles', 'passengers', 'fare', 'carrier_lg', 'fare_lg', 'carrier_low', 'fare_low']
flight_data = flight_data[keep]

In [ ]:
flight_data

In [ ]:
#removing years before 2018
flight_data = flight_data[flight_data['Year']>2017]
flight_data.dropna()
flight_data

In [ ]:
#reordering columns
cols = list(flight_data.columns.values)
cols
cols = ['Year','airport_1','airport_2','fare', 'nsmiles', 'carrier_lg', 'fare_lg', 'carrier_low', 'fare_low', 'city1', 'city2', 'passengers', 'quarter']
flight_data= flight_data[cols]
flight_data

In [ ]:
airports = flight_data['airport_1'].tolist() + flight_data['airport_2'].tolist()
len(airports),len(set(airports))
#176 unique airports found

In [ ]:
#reverse all routes and add back to original data
copy = flight_data.copy()
copy1 = flight_data.copy()
col_list = list(copy1)
#swap position of airport_1 and airport_2 column
col_list[1], col_list[2] = col_list[2], col_list[1]
copy.columns = col_list
#swap airport_1 and airport_2 column headers to swap data in columns
cols = ['Year','airport_1','airport_2','fare', 'nsmiles', 'carrier_lg', 'fare_lg', 'carrier_low', 'fare_low', 'city1', 'city2', 'passengers', 'quarter']
copy = copy[cols]
flight_data = pd.concat([copy,copy1])
flight_data

In [ ]:
#merging rows with identical routes
flight_data= flight_data.groupby(['airport_1', 'airport_2'], ).agg({'fare': 'mean', 'nsmiles':'mean', 'carrier_lg': 'first', 'fare_lg': 'mean',
                                                                     'carrier_low': 'first', 'fare_low': 'mean', 'city1': 'first', 'city2': 'first'}).reset_index()

In [ ]:
#creating graph
graph_dict = {}
#iterate through every route
for index, row in flight_data.iterrows():
  #add airport_2 to adjacency of airport_1
  if row['airport_1'] not in graph_dict.keys():
    graph_dict[row['airport_1']] = [row['airport_2']]
  else:
    graph_dict[row['airport_1']].append(row['airport_2'])

In [ ]:
#creating weighted graph, same as before with weights
weighted_graph_dict = {}
for index, row in flight_data.iterrows():
  if row['airport_1'] not in weighted_graph_dict.keys():
    #add airport_2 and weight to adjacency of airport_1
    weighted_graph_dict[row['airport_1']] = {row['airport_2'] : row['fare']}
  else:
    weighted_graph_dict[row['airport_1']][row['airport_2']] = row['fare']

In [ ]:
#creating weighted graph, with a constant added to the weights
layover_graph_dict = {}
layover_cost = 80
for index, row in flight_data.iterrows():
  if row['airport_1'] not in layover_graph_dict.keys():
    #add airport_2 and weight to adjacency of airport_1
    layover_graph_dict[row['airport_1']] = {row['airport_2'] : row['fare']+layover_cost}
  else:
    layover_graph_dict[row['airport_1']][row['airport_2']] = row['fare']+layover_cost

In [ ]:
#top airports by connections
len_dict = {}
for key,value in graph_dict.items():
  len_dict[key] = len(value)
dict(reversed(list(dict(sorted(len_dict.items(), key=lambda item: item[1])).items())))

In [ ]:
#lookup airport connections
lookup = 'SAN'
len(graph_dict[lookup])

In [ ]:
#bfs

#initialize visited nodes set
visited = set()
def bfs(start, end, graph):
  #create queue with starting node
  queue = deque()
  queue.append([start])
  #iterate while queue contains elements
  while queue:
    #take out last explored path from queue
    path = queue.popleft()
    node = path[-1]
    if node not in visited:
      visited.add(node)
      #check if the path ends with goal node
      if node == end:
        return path
      #traverse to all neighbors and add paths to stack
      for i in graph_dict[node]:
        temp_path = list(path)
        temp_path.append(i)
        queue.append(temp_path)
  #false if no path exists
  return ['false']

In [ ]:
#dfs

#initialize visited nodes set and stack
visited = set()
stack = deque()
def dfs(visited, graph, node, goal):
    #add current node to stack
    stack.append(node)
    #check if current node is goal, then return stack
    if node == goal:
        return stack
    #check to make sure node not already visited
    if node not in visited:
        #add node to visited set
        visited.add(node)
        #if next nodes exist, traverse to them
        if graph[node]:
          #randomly pick a neighbor to traverse to
          random.shuffle(graph[node])
          #for all neigbors, recursively run dfs
          for neighbor in graph[node]:
              if neighbor not in visited:
                path = dfs(visited, graph, neighbor, goal)
              #return stack if goal found
                if path:
                  return path
              #if no more nodes to traverse, pop path
                else:
                  stack.pop()
        #if no more nodes to traverse, pop path
        else:
          stack.pop()
    return None

In [ ]:
#djikstras

def djikstras(start, end, graph):
  #initialize distances as infinity, except start
  distances = {node : float('inf') for node in graph}
  distances[start] = 0
  #initialze dictionary to keep track of previous node in path
  last_node = {node: None for node in graph}
  #create priority queue to traverse by dfs through graph
  queue = [(0, start)]

  while queue:
    #pop least weight path
    temp_distance, temp_node = heapq.heappop(queue)
    #traverse to neighbors
    for next,weight in graph[temp_node].items():
      #calculate tentative weight for neighbor node
      next_distance_temp = temp_distance + weight
      #if path weight less than current, overwrite current weight
      if next_distance_temp < distances[next]:
        distances[next] = next_distance_temp
        last_node[next] = temp_node
        #push path to priority queue
        heapq.heappush(queue, (next_distance_temp, next))

    #backtrack from node to find path
    exist = last_node[end]
    path = [end]
    while exist:
      path.append(exist)
      exist = last_node[exist]
    path.reverse()

  return distances, last_node, path

In [ ]:
#find cost of a given path

def cost_from_path(path):
  cost = 0
  for i in range(len(path)-1):
    j = i+1
    cost+=weighted_graph_dict[path[i]][path[j]]
  return cost

In [ ]:
#testing bfs
starting = 'BLV'
ending = 'SAN'

visited = set()
t0 = time.time()
results = bfs(starting, ending, graph_dict)
t1 = time.time()

print(results)
print('Time:', t1-t0, 'seconds')
print("Length:", len(results)-1)
print('Cost: ', cost_from_path(results))

In [ ]:
#testing dfs
starting = 'BLV'
ending = 'SAN'

visited = set()
stack = []
t0 = time.time()
results= dfs(visited, graph_dict, starting, ending)
t1 = time.time()

print(results)
print('Time:', t1-t0, 'seconds')
print("Length:", len(results)-1)
print('Cost: ', cost_from_path(results))

In [ ]:
#testing djikstras
starting = 'ORD'
ending = 'SEA'

t0 = time.time()
distances, last_node, path = djikstras(starting, ending, weighted_graph_dict)
t1 = time.time()

print(path)
print('Cost:', distances[ending])
print('Time:', t1-t0, 'seconds')
print('Length:', len(path)-1)

In [ ]:
#testing modified djikstras
starting = 'ATL'
ending = 'SAN'

t0 = time.time()
distances, last_node, path = djikstras(starting, ending, layover_graph_dict)
t1 = time.time()

print(path)
print('Cost:', cost_from_path(path))
print('Time:', t1-t0, 'seconds')
print('Length:', len(path)-1)

In [ ]:
#testing djikstras with vs without mod
starting = 'BNA'
ending = 'RNO'

t0 = time.time()
distances, last_node, path = djikstras(starting, ending, weighted_graph_dict)
t1 = time.time()

print(path)
print('Cost:', distances[ending])
print('Time:', t1-t0, 'seconds')
print('Length:', len(path)-1)

t0 = time.time()
distances, last_node, path = djikstras(starting, ending, layover_graph_dict)
t1 = time.time()

print(path)
print('Cost:', distances[ending] - (len(path)-1)*80)
print('Time:', t1-t0, 'seconds')
print('Length:', len(path)-1)

In [ ]:
algos = ['DFS', 'BFS', 'Djikstras', 'Modified Djikstras']
times = [[],[],[],[]]
answers = [[],[],[],[]]
costs = [[],[],[],[]]
for i in range(25000):
  keys = list(set(graph_dict.keys()))
  start = random.choice(keys)
  end = random.choice(keys)
  while end==start:
    end = random.choice(keys)

  visited = set()
  t0bfs = time.time()
  results = bfs(start, end, graph_dict)
  t1bfs = time.time()
  answers[1].append(len(results)-1)
  costs[1].append(cost_from_path(results))

  visited = set()
  stack = deque()
  t0dfs = time.time()
  results= dfs(visited, graph_dict, start, end)
  t1dfs = time.time()
  answers[0].append(len(results)-1)
  costs[0].append(cost_from_path(results))

  t0djikstras = time.time()
  distances, last_node, results = djikstras(start, end, weighted_graph_dict)
  t1djikstras = time.time()
  answers[2].append(len(results)-1)
  costs[2].append(cost_from_path(results))

  t0djikstrasmod = time.time()
  distances, last_node, results = djikstras(start, end, layover_graph_dict)
  t1djikstrasmod = time.time()
  answers[3].append(len(results)-1)
  costs[3].append(cost_from_path(results))


  times[0].append(t1dfs-t0dfs)
  times[1].append(t1bfs-t0bfs)
  times[2].append(t1djikstras-t0djikstras)
  times[3].append(t1djikstrasmod-t0djikstrasmod)

timesmean = [[],[],[],[]]
timessum = [[],[],[],[]]
timesmean[0] = mean(times[0])
timesmean[1] = mean(times[1])
timesmean[2] = mean(times[2])
timesmean[3] = mean(times[3])
timessum[0] = sum(times[0])
timessum[1] = sum(times[1])
timessum[2] = sum(times[2])
timessum[3] = sum(times[3])

In [ ]:
timealgosumdf = pd.DataFrame({'Algorithms': algos, 'Times': timessum})

def addlabels(x,y):
    for i in range(len(x)):
        plt.text(i, y[i], y[i], ha = 'center')

fig, ax = plt.subplots()

ax.barh(timealgosumdf.Algorithms, timealgosumdf.Times)
ax.set_xlabel('Total Time')
ax.set_ylabel('Algorithm Used')

plt.show()
print(timessum)

In [ ]:
timealgomeandf = pd.DataFrame({'Algorithms': algos, 'Times': timesmean})

fig, ax = plt.subplots()

ax.barh(timealgomeandf.Algorithms, timealgomeandf.Times)
ax.set_xlabel('Mean Time (s)')
ax.set_ylabel('Algorithm Used')

plt.show()
print(timesmean)

In [ ]:
answersmean = [[],[],[],[]]
answersmean[0] = mean(answers[0])
answersmean[1] = mean(answers[1])
answersmean[2] = mean(answers[2])
answersmean[3] = mean(answers[3])

anslen = pd.DataFrame({'Algorithms': algos, 'Lens': answersmean})

fig, ax = plt.subplots()

ax.barh(anslen.Algorithms, anslen.Lens)
ax.set_xlabel('Mean Length (steps)')
ax.set_ylabel('Algorithm Used')

plt.show()
print(answersmean)

In [ ]:
answersmean = [[],[],[]]
answersmean[0] = mean(answers[1])
answersmean[1] = mean(answers[2])
answersmean[2] = mean(answers[3])

anslen = pd.DataFrame({'Algorithms': algos[1:4], 'Lens': answersmean})

fig, ax = plt.subplots()

ax.barh(anslen.Algorithms, anslen.Lens)
ax.set_xlabel('Mean Length (steps)')
ax.set_ylabel('Algorithm Used')

plt.show()
print(answersmean)

In [ ]:
costsmean = [[],[],[],[]]
costsmean[0] = mean(costs[0])
costsmean[1] = mean(costs[1])
costsmean[2] = mean(costs[2])
costsmean[3] = mean(costs[3])
costdf = pd.DataFrame({'Algorithms': algos, 'means': costsmean})

fig, ax = plt.subplots()

ax.barh(costdf.Algorithms, costdf.means)
ax.set_xlabel('Mean Cost (dollars)')
ax.set_ylabel('Algorithm Used')

plt.show()
print(costsmean)

In [ ]:
costsmean = [[],[],[]]
costsmean[0] = mean(costs[1])
costsmean[1] = mean(costs[2])
costsmean[2] = mean(costs[3])
costdf = pd.DataFrame({'Algorithms': algos[1:4], 'means': costsmean})

fig, ax = plt.subplots()

ax.barh(costdf.Algorithms, costdf.means)
ax.set_xlabel('Mean Cost (dollars)')
ax.set_ylabel('Algorithm Used')

plt.show()
print(costsmean)